# Aula 3, Overfitting e underfitting

Notebook executável que acompanha a aula [03-overfitting-underfitting.md](../../lessons/modulo-02-fundamentos-ml/03-overfitting-underfitting.md).

## O que vamos fazer aqui

Vamos ver, com as próprias mãos, o equilíbrio mais importante do Machine Learning.
Ajustando polinômios de grau crescente aos mesmos dados, vamos observar o erro de
treino cair sempre, enquanto o erro de teste desenha um U, que é a assinatura do
overfitting.

A relação verdadeira que geramos é levemente curva (grau 2), então esperamos que o
melhor equilíbrio apareça por volta desse grau. O notebook usa apenas numpy, com
gráfico opcional.

## Dados e divisão treino/teste

Geramos dados a partir de uma relação curva entre horas e nota, com ruído, e
separamos em treino e teste. O modelo só enxerga o treino no ajuste, e o teste serve
para medir a generalização.

In [ ]:
import numpy as np

rng = np.random.default_rng(2)

def relacao_real(x):
    return 4 + 1.5 * x - 0.08 * x**2

horas = rng.uniform(0, 10, size=60)
notas = relacao_real(horas) + rng.normal(0, 0.8, size=60)

indices = rng.permutation(len(horas))
treino, teste = indices[:40], indices[40:]
x_tr, y_tr = horas[treino], notas[treino]
x_te, y_te = horas[teste], notas[teste]
print('treino:', len(x_tr), '| teste:', len(x_te))

Separar treino de teste é a regra de ouro, nunca avaliar o modelo nos mesmos
dados em que ele treinou. São 40 exemplos para treinar e 20 para medir o desempenho
em dados novos.

## Erro de treino e de teste por grau

Para cada grau de 1 a 10, ajustamos um polinômio aos dados de treino com
`np.polyfit` e medimos o erro nos dois conjuntos. Acompanhe as duas colunas conforme
o grau aumenta.

In [ ]:
def mse(y_real, y_prev):
    return np.mean((y_real - y_prev) ** 2)

linhas = []
print(f"{'grau':>4} {'erro treino':>12} {'erro teste':>12}")
for grau in range(1, 11):
    coef = np.polyfit(x_tr, y_tr, grau)
    erro_tr = mse(y_tr, np.polyval(coef, x_tr))
    erro_te = mse(y_te, np.polyval(coef, x_te))
    linhas.append((grau, erro_tr, erro_te))
    print(f"{grau:>4} {erro_tr:>12.3f} {erro_te:>12.3f}")

melhor = min(linhas, key=lambda t: t[2])
print(f"\nMenor erro de teste no grau {melhor[0]} (erro {melhor[2]:.3f}).")

O erro de treino cai de forma quase monótona, porque o polinômio fica cada
vez mais livre para passar perto dos pontos de treino. O erro de teste, porém, cai no
começo e volta a subir, sinalizando o overfitting. O menor erro de teste costuma cair
no grau 2, que é exatamente o grau da relação que geramos.

## Gráfico opcional

O gráfico abaixo deixa o U do erro de teste visível, contrastando com a queda
contínua do erro de treino. Roda se o matplotlib estiver instalado.

In [ ]:
try:
    import matplotlib.pyplot as plt

    graus = [l[0] for l in linhas]
    erros_tr = [l[1] for l in linhas]
    erros_te = [l[2] for l in linhas]
    plt.plot(graus, erros_tr, marker='o', label='erro treino')
    plt.plot(graus, erros_te, marker='o', label='erro teste')
    plt.xlabel('grau do polinômio')
    plt.ylabel('MSE')
    plt.legend()
    plt.show()
except ImportError:
    print('matplotlib não instalado, pulando o gráfico (veja docs/setup.md).')

Para o projeto da aula, a partir da tabela ou do gráfico, aponte qual grau
sofre underfitting (erro alto nos dois), qual sofre overfitting (erro baixo no treino
e alto no teste) e qual oferece o melhor equilíbrio, justificando com os números.